In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import re

In [ ]:
# Find all experiment result files
build_dir = Path('../build')

dijkstra_files = sorted(build_dir.glob('*dijkstra*.csv'), key=lambda x: x.stat().st_mtime, reverse=True)
deltastep_files = sorted(build_dir.glob('*deltastep*.csv'), key=lambda x: x.stat().st_mtime, reverse=True)
bellmanford_files = sorted(build_dir.glob('*bellmanford*.csv'), key=lambda x: x.stat().st_mtime, reverse=True)
nearfar_files = sorted(build_dir.glob('*nearfar*.csv'), key=lambda x: x.stat().st_mtime, reverse=True)

print(f"Found {len(dijkstra_files)} Dijkstra files")
print(f"Found {len(deltastep_files)} Deltastep files")
print(f"Found {len(bellmanford_files)} BellmanFord files")
print(f"Found {len(nearfar_files)} NearFar files")

In [ ]:
# Extract query hash and device hash from filename
# Format: timestamp_queryhash_devicehash_[params_]algorithm.csv
def get_hashes(filename):
    match = re.search(r'_(\w{8})_(\w{8})_', filename)
    if match:
        return match.group(1), match.group(2)
    return None, None

# Find the most recent Dijkstra file with any device (we ignore device for CPU)
dijkstra_file = dijkstra_files[0] if dijkstra_files else None
if not dijkstra_file:
    raise ValueError("No Dijkstra file found")

dijkstra_query_hash, _ = get_hashes(dijkstra_file.name)
print(f"Using Dijkstra: {dijkstra_file.name} (query_hash: {dijkstra_query_hash})")

# Group GPU experiment files by device hash
device_experiments = {}

for delta_f in deltastep_files:
    delta_query_hash, delta_device_hash = get_hashes(delta_f.name)
    if delta_query_hash == dijkstra_query_hash and delta_device_hash:
        if delta_device_hash not in device_experiments:
            device_experiments[delta_device_hash] = {'deltastep': None, 'bellmanford': None, 'nearfar': None}
        if device_experiments[delta_device_hash]['deltastep'] is None:
            device_experiments[delta_device_hash]['deltastep'] = delta_f

for bellman_f in bellmanford_files:
    bellman_query_hash, bellman_device_hash = get_hashes(bellman_f.name)
    if bellman_query_hash == dijkstra_query_hash and bellman_device_hash:
        if bellman_device_hash not in device_experiments:
            device_experiments[bellman_device_hash] = {'deltastep': None, 'bellmanford': None, 'nearfar': None}
        if device_experiments[bellman_device_hash]['bellmanford'] is None:
            device_experiments[bellman_device_hash]['bellmanford'] = bellman_f

for nearfar_f in nearfar_files:
    nearfar_query_hash, nearfar_device_hash = get_hashes(nearfar_f.name)
    if nearfar_query_hash == dijkstra_query_hash and nearfar_device_hash:
        if nearfar_device_hash not in device_experiments:
            device_experiments[nearfar_device_hash] = {'deltastep': None, 'bellmanford': None, 'nearfar': None}
        if device_experiments[nearfar_device_hash]['nearfar'] is None:
            device_experiments[nearfar_device_hash]['nearfar'] = nearfar_f

# Filter to only devices with all algorithms
device_experiments = {
    device_hash: files 
    for device_hash, files in device_experiments.items() 
    if files['deltastep'] is not None and files['bellmanford'] is not None and files['nearfar'] is not None
}

print(f"\nFound experiments for {len(device_experiments)} device(s):")
for device_hash, files in device_experiments.items():
    print(f"  Device {device_hash}:")
    print(f"    - Deltastep: {files['deltastep'].name}")
    print(f"    - BellmanFord: {files['bellmanford'].name}")
    print(f"    - NearFar: {files['nearfar'].name}")

In [ ]:
# Load Dijkstra data once (CPU baseline)
df_dijkstra = pd.read_csv(dijkstra_file)
print(f"Dijkstra: {len(df_dijkstra)} queries")

# Load data for each device and create merged datasets
device_data = {}

for device_hash, files in device_experiments.items():
    df_deltastep = pd.read_csv(files['deltastep'])
    df_bellmanford = pd.read_csv(files['bellmanford'])
    df_nearfar = pd.read_csv(files['nearfar'])
    
    print(f"\nDevice {device_hash}:")
    print(f"  Deltastep: {len(df_deltastep)} queries")
    print(f"  BellmanFord: {len(df_bellmanford)} queries")
    print(f"  NearFar: {len(df_nearfar)} queries")
    
    # Merge all datasets for this device
    df_merged = pd.merge(
        df_dijkstra,
        df_deltastep,
        on=['from_node_id', 'to_node_id', 'rank'],
        suffixes=('_dijkstra', '_deltastep')
    )
    
    df_merged = pd.merge(
        df_merged,
        df_bellmanford,
        on=['from_node_id', 'to_node_id', 'rank']
    )
    df_merged.rename(columns={'distance': 'distance_bellmanford', 'time': 'time_bellmanford'}, inplace=True)
    
    df_merged = pd.merge(
        df_merged,
        df_nearfar,
        on=['from_node_id', 'to_node_id', 'rank']
    )
    df_merged.rename(columns={'distance': 'distance_nearfar', 'time': 'time_nearfar'}, inplace=True)
    
    device_data[device_hash] = {
        'dijkstra': df_dijkstra,
        'deltastep': df_deltastep,
        'bellmanford': df_bellmanford,
        'nearfar': df_nearfar,
        'merged': df_merged
    }
    
    print(f"  Merged: {len(df_merged)} queries")

In [ ]:
# Check for distance errors for each device
for device_hash, data in device_data.items():
    print(f"\n{'='*60}")
    print(f"Device {device_hash}")
    print('='*60)
    
    df_merged = data['merged']
    
    errors_dijkstra_deltastep = df_merged[df_merged['distance_dijkstra'] != df_merged['distance_deltastep']]
    errors_dijkstra_bellmanford = df_merged[df_merged['distance_dijkstra'] != df_merged['distance_bellmanford']]
    errors_dijkstra_nearfar = df_merged[df_merged['distance_dijkstra'] != df_merged['distance_nearfar']]
    
    if len(errors_dijkstra_deltastep) > 0:
        print(f"⚠️  Found {len(errors_dijkstra_deltastep)} distance mismatches between Dijkstra and Delta-stepping!")
        print(errors_dijkstra_deltastep[['from_node_id', 'to_node_id', 'distance_dijkstra', 'distance_deltastep']].head(10))
    else:
        print("✓ All distances match between Dijkstra and Delta-stepping")
    
    if len(errors_dijkstra_bellmanford) > 0:
        print(f"⚠️  Found {len(errors_dijkstra_bellmanford)} distance mismatches between Dijkstra and Bellman-Ford!")
        print(errors_dijkstra_bellmanford[['from_node_id', 'to_node_id', 'distance_dijkstra', 'distance_bellmanford']].head(10))
    else:
        print("✓ All distances match between Dijkstra and Bellman-Ford")
    
    if len(errors_dijkstra_nearfar) > 0:
        print(f"⚠️  Found {len(errors_dijkstra_nearfar)} distance mismatches between Dijkstra and Near-Far!")
        print(errors_dijkstra_nearfar[['from_node_id', 'to_node_id', 'distance_dijkstra', 'distance_nearfar']].head(10))
    else:
        print("✓ All distances match between Dijkstra and Near-Far")

In [ ]:
# Histogram comparing execution times for each device
for device_hash, data in device_data.items():
    print(f"\n{'='*60}")
    print(f"Device {device_hash}")
    print('='*60)
    
    df_dijkstra = data['dijkstra']
    df_deltastep = data['deltastep']
    df_bellmanford = data['bellmanford']
    df_nearfar = data['nearfar']
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    bin_width = np.max([df_dijkstra['time'].max(), df_deltastep['time'].max(), df_bellmanford['time'].max(), df_nearfar['time'].max()])/20
    
    sns.histplot(df_dijkstra['time'], label='Dijkstra', alpha=0.5, ax=ax, binwidth=bin_width)
    sns.histplot(df_deltastep['time'], label='Delta-stepping', alpha=0.5, ax=ax, binwidth=bin_width)
    sns.histplot(df_bellmanford['time'], label='Bellman-Ford', alpha=0.5, ax=ax, binwidth=bin_width)
    sns.histplot(df_nearfar['time'], label='Near-Far', alpha=0.5, ax=ax, binwidth=bin_width)
    
    ax.set_xlabel('Time (μs)')
    ax.set_ylabel('Count')
    ax.set_title(f'Execution Time Distribution - Device {device_hash}')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Box plot by query rank for each device
for device_hash, data in device_data.items():
    print(f"\n{'='*60}")
    print(f"Device {device_hash}")
    print('='*60)
    
    df_merged = data['merged']
    
    plot_data = []
    for _, row in df_merged.iterrows():
        plot_data.append({'rank': row['rank'], 'time': row['time_dijkstra']/10**6, 'algorithm': 'Dijkstra'})
        plot_data.append({'rank': row['rank'], 'time': row['time_deltastep']/10**6, 'algorithm': 'Delta-stepping'})
        plot_data.append({'rank': row['rank'], 'time': row['time_bellmanford']/10**6, 'algorithm': 'Bellman-Ford'})
        plot_data.append({'rank': row['rank'], 'time': row['time_nearfar']/10**6, 'algorithm': 'Near-Far'})
    
    df_plot = pd.DataFrame(plot_data)
    
    fig, ax = plt.subplots(figsize=(14, 6))
    sns.boxplot(data=df_plot, x='rank', y='time', hue='algorithm', ax=ax)
    ax.set_xlabel('Query Rank Bin')
    ax.set_ylabel('Time (s)')
    plt.yscale('log')
    ax.set_title(f'Execution Time by Query Rank - Device {device_hash}')
    plt.tight_layout()
    plt.show()